# ChessDorrie — Tal Troll Analyzer (Colab launcher)

**How to launch:** `Runtime → Change runtime type → T4 GPU` (or A100/L4 if
available — T4 is enough for inference), then `Runtime → Run all`.
After ~1–3 minutes the last cell prints a public URL — open it in a new tab.

The setup cell calls `scripts/colab_setup.sh`, which:
* installs Stockfish + Python deps + torch
* installs **Lc0 with the CUDA backend** (uses `scripts/install_lc0_cuda.sh`
  — if your GPU isn't picked up it falls back gracefully to softmax-Stockfish)
* downloads the three Maia weights (1100 / 1500 / 1900)
* optionally pulls a **multi-million-game Lichess dump** for training
  if you uncomment the `--with-data` flag below (~28 GB, 15–30 min)
* runs a smoke-verify step and prints `ALL READY` if everything works

**Session goes idle after ~90 min.** To bring the tunnel back, re-run the
last two cells (launch server + tunnel).

**If something looks wrong:** the very last cell tails the server log.

## 1. Clone / pull the repo

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/omarnuri/chessdorrie'
REPO_BRANCH = 'claude/stoic-hypatia-U7ain'
REPO_DIR = '/content/ChessDorrie'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False)
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
print('Branch:', subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip())

## 2. One-shot setup (apt + pip + Lc0 + Maia + smoke verify)

In [ ]:
# Inference-only setup (default). For training, append:
#     --with-data 2024-09     (28 GB monthly dump, 15-30 min in Colab)
!bash scripts/colab_setup.sh

## 3. (Optional) Training data + mining

Uncomment the line in the next cell to pull a Lichess monthly dump
(~28 GB) and mine traps from it. Skip if you only want inference.

In [ ]:
# Uncomment to download a recent month and re-mine the trap DB.
# Picks one from https://database.lichess.org/  (a recent month is ~28 GB)
# !bash scripts/colab_setup.sh --with-data 2024-09
# !python -m data.mine_lichess --pgn data/dumps/lichess_2024-09.pgn.zst --max-games 1000000 --out data/mined_traps.json

## 4. (Optional) Train a trap-policy network (A100/H100 only)

Skeleton commands. Expect ~6–12 h on A100, ~2–4 h on H100 for
3 epochs on ~5M positions. See `scripts/train_trap_policy.py`.

In [ ]:
# Uncomment the three steps to filter, tensorize, and train.
# !python scripts/dataset_filter_traps.py --pgn data/dumps/lichess_2024-09.pgn.zst --out data/trap_games.pgn
# !python scripts/dataset_to_tensors.py --pgn data/trap_games.pgn --out data/tensors/ --max-positions 5000000
# !python scripts/train_trap_policy.py --data data/tensors/ --out trained/ --init-from weights/maia-1500.pb.gz --batch-size 256 --epochs 3
# When done, the resulting trained/best.pt can be served by setting TRAP_MODEL=trained/best.pt before launching.

## 5. Optional: mine a Lichess monthly dump for additional traps

A single month is ~10 GB. The `--max-games` flag caps the scan; 100k
gives a useful trap corpus in ~5 minutes on Colab's CPU.

In [ ]:
# Uncomment to run. Pick a recent month from https://database.lichess.org/
# !pip install -q zstandard
# !python -m data.mine_lichess --month 2024-09 --max-games 100000 --out data/mined_traps.json

## 6. Launch the server

In [ ]:
import subprocess, time, os, signal, glob
# Kill any previous server.
subprocess.run(['pkill', '-f', 'app.server'], check=False)
time.sleep(1)
env = os.environ.copy()
env['PYTHONPATH'] = os.getcwd()
env['CD_PORT'] = '8000'
env['CD_HOST'] = '0.0.0.0'
env['CD_THREADS'] = '4'
# If a trained checkpoint exists, serve it as the human model
trained = sorted(glob.glob('trained/*.pt'))
if trained:
    env['TRAP_MODEL'] = trained[-1]
    print('Using trained model:', env['TRAP_MODEL'])
server = subprocess.Popen(
    ['python', '-m', 'app.server'],
    env=env, stdout=open('/tmp/cd_server.log', 'w'), stderr=subprocess.STDOUT,
)
print('Server PID:', server.pid)
for _ in range(20):
    time.sleep(0.5)
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=1).read()
        print('Server is up.')
        break
    except Exception:
        continue
else:
    print('Server did not come up — check /tmp/cd_server.log')

## 7. Expose via Cloudflare quick tunnel

No login required; the URL is ephemeral and dies with the kernel.

In [ ]:
import os, subprocess, re, time
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        public_url = m.group(1)
        break
from IPython.display import HTML, display
if public_url:
    display(HTML(f'<h2>ChessDorrie is live → <a href="{public_url}" target="_blank">{public_url}</a></h2>'))
else:
    print('Cloudflared did not return a URL. Check the cell output for errors.')

## Tail the server log (handy if something looks wrong)

In [ ]:
!tail -30 /tmp/cd_server.log